# Laboratorio L5 - Pulire i dati

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessario: `tragitti_esempio.csv` (oppure il CSV del questionario della classe: in questo caso adattare i nomi delle colonne e le correzioni).

Regola: il file originale non si modifica. Si lavora su una copia in memoria e alla fine si salva un nuovo file, `tragitti_puliti_classe.csv`. Ogni operazione va annotata nel registro della pulizia (ultima sezione).

In [1]:
import pandas as pd

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

originale = pd.read_csv("tragitti_esempio.csv")
dati = originale.copy()       # copia di lavoro: l'originale resta intatto
print(dati.shape)
dati.head(10)

(153, 7)


,id,inviato_il,anno_corso,mezzo,distanza_km,tempo_min,fascia_partenza
0,1,2026-10-06 08:50:44,4,bus,10.0,46,7:00-7:30
1,2,2026-10-06 08:10:32,5,auto,8.6,27,7:00-7:30
2,3,2026-10-06 08:05:40,5,bicicletta,4.7,19,7:30-7:45
3,4,2026-10-06 08:39:44,4,monopattino elettrico,3.7,20,dopo le 7:45
4,5,2026-10-06 08:55:09,3,piedi,0.8,11,dopo le 7:45
5,6,2026-10-06 08:35:53,5,auto,12.6,34,7:00-7:30
6,7,2026-10-06 08:06:09,4,monopattino,"1,2",3,dopo le 7:45
7,8,2026-10-06 08:05:51,4,bicicletta,5.3,22,7:30-7:45
8,9,2026-10-06 08:41:20,5,monopattino,3.5,9,7:30-7:45
9,10,2026-10-06 08:50:57,3,treno,30.0,61',7:00-7:30


In [2]:
dati.info()

<class 'pandas.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               153 non-null    int64
 1   inviato_il       153 non-null    str  
 2   anno_corso       153 non-null    int64
 3   mezzo            153 non-null    str  
 4   distanza_km      152 non-null    str  
 5   tempo_min        149 non-null    str  
 6   fascia_partenza  151 non-null    str  
dtypes: int64(2), str(5)
memory usage: 8.5 KB


## 1. Uniformare il testo: il mezzo di trasporto (esercizio 1)

- `str.strip()`: toglie gli spazi all'inizio e alla fine
- `str.lower()`: trasforma in minuscolo
- `replace(dizionario)`: sostituisce ogni valore che compare come chiave del dizionario con il valore associato

Un dizionario Python si scrive tra graffe: `{"vecchio": "nuovo", ...}`.

In [3]:
dati["mezzo"] = dati["mezzo"].str.strip().str.lower()
dati["mezzo"].value_counts()

mezzo
bus                      45
piedi                    26
auto                     24
bici                     18
monopattino              13
treno                     6
a piedi                   4
in auto                   3
pullman                   3
autobus                   3
treno + bus               3
bicicletta                2
monopattino elettrico     2
macchina                  1
Name: count, dtype: int64

Completare il dizionario `correzioni` in modo che restino solo i sei mezzi del dizionario dei dati: `piedi`, `bici`, `monopattino`, `bus`, `auto`, `treno`. Per `treno + bus` si applica la regola del questionario: conta il mezzo su cui si passa più tempo (il treno).

In [4]:
correzioni = {
    "autobus": "bus",
    "pullman": "bus",
    "a piedi": "piedi",
    "bicicletta": "bici",
    "macchina": "auto",
    "in auto": "auto",
    "monopattino elettrico": "monopattino",
    "treno + bus": "treno",
}
dati["mezzo"] = dati["mezzo"].replace(correzioni)
print(dati["mezzo"].value_counts())
controlla(set(dati["mezzo"].unique()) == {"piedi", "bici", "monopattino", "bus", "auto", "treno"},
          "Controllare quali valori compaiono ancora oltre ai sei ammessi.")

mezzo
bus            51
piedi          30
auto           28
bici           20
monopattino    15
treno           9
Name: count, dtype: int64
Corretto.


## 2. Numeri scritti come testo (esercizio 2)

Distanza: alcune risposte usano la virgola decimale (`1,2`).

- `astype(str)`: tratta ogni valore come testo
- `str.replace(",", ".")`: sostituisce la virgola con il punto
- `pd.to_numeric(..., errors="coerce")`: converte in numero; ciò che non è convertibile diventa `NaN`

In [5]:
dati["distanza_km"] = pd.to_numeric(dati["distanza_km"].astype(str).str.replace(",", "."), errors="coerce")
print(dati["distanza_km"].dtype, "- valori mancanti:", dati["distanza_km"].isna().sum())

float64 - valori mancanti: 1


Tempo: alcune risposte contengono l'unità (`45 min`, `45 minuti`, `45'`). Si estrae la parte numerica.

- `str.extract(r"(\d+)")`: estrae la prima sequenza di cifre del testo; `\d` indica una cifra, `+` una o più, le parentesi indicano la parte da estrarre. È un'espressione regolare, un linguaggio per descrivere schemi di testo
- `[0]`: `extract` restituisce una tabella; si prende la sua prima (e unica) colonna

In [6]:
print("prima:", dati["tempo_min"].astype(str).str.contains("[a-z']", regex=True).sum(), "risposte con testo")
dati["tempo_min"] = pd.to_numeric(dati["tempo_min"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
print(dati["tempo_min"].dtype, "- valori mancanti:", dati["tempo_min"].isna().sum())
controlla(dati["tempo_min"].dtype.kind == "f" and dati["distanza_km"].dtype.kind == "f")

prima: 10 risposte con testo
float64 - valori mancanti: 4
Corretto.


## 3. Duplicati (esercizio 3)

Un modulo online può ricevere due volte la stessa risposta (doppio clic, invio ripetuto). Le risposte duplicate hanno le stesse risposte ma data di invio e `id` diversi, quindi il confronto va fatto solo sulle colonne delle risposte (`subset`).

- `duplicated(subset=[...])`: `True` per ogni riga uguale a una riga precedente nelle colonne indicate
- `keep=False`: segna tutte le copie, anche la prima, per poterle esaminare
- `drop_duplicates(subset=[...])`: elimina le copie successive alla prima

In [7]:
risposte = ["anno_corso", "mezzo", "distanza_km", "tempo_min", "fascia_partenza"]
print("righe duplicate:", dati.duplicated(subset=risposte).sum())
dati[dati.duplicated(subset=risposte, keep=False)].sort_values("distanza_km")

righe duplicate: 3


,id,inviato_il,anno_corso,mezzo,distanza_km,tempo_min,fascia_partenza
24,25,2026-10-06 08:58:14,4,bici,3.9,14.0,7:30-7:45
115,116,2026-10-06 08:23:26,4,bici,3.9,14.0,7:30-7:45
131,132,2026-10-06 08:29:20,5,bici,4.7,18.0,dopo le 7:45
10,11,2026-10-06 08:37:06,5,bici,4.7,18.0,dopo le 7:45
89,90,2026-10-06 08:56:31,3,bus,4.8,27.0,7:00-7:30
19,20,2026-10-06 08:56:43,3,bus,4.8,27.0,7:00-7:30


**Domanda**: due studenti diversi potrebbero dare esattamente le stesse risposte? Con quali colonne del dataset lo si potrebbe verificare? Qui si considerano duplicati.

Risposta:

In [8]:
dati = dati.drop_duplicates(subset=risposte)
print(dati.shape)
controlla(len(dati) == 150)

(150, 7)
Corretto.


## 4. Valori impossibili e valori anomali (esercizio 4)

- valori impossibili: violano le regole del dizionario dei dati (distanza negativa o zero, tempo zero); sono sicuramente errori
- valori anomali (outlier): molto lontani dagli altri; possono essere errori o casi reali

Criterio dei quartili: è anomalo un valore minore di Q1 - 1,5 x IQR o maggiore di Q3 + 1,5 x IQR (lo stesso usato dai baffi del boxplot).

In [9]:
print(dati[(dati["distanza_km"] <= 0) | (dati["tempo_min"] <= 0)][["id", "mezzo", "distanza_km", "tempo_min"]])

      id mezzo  distanza_km  tempo_min
77    78  auto         -3.0       28.0
114  115   bus         10.3        0.0


In [10]:
def anomali(colonna):
    q1, q3 = colonna.quantile(0.25), colonna.quantile(0.75)
    iqr = q3 - q1
    return (colonna < q1 - 1.5 * iqr) | (colonna > q3 + 1.5 * iqr)

dati[anomali(dati["distanza_km"])][["id", "mezzo", "distanza_km", "tempo_min"]].sort_values("distanza_km")

,id,mezzo,distanza_km,tempo_min
98,99,treno,23.4,55.0
20,21,treno,23.9,52.0
133,134,treno,24.1,53.0
120,121,treno,28.4,60.0
9,10,treno,30.0,61.0
130,131,treno,31.1,68.0
116,117,treno,35.0,75.0
43,44,bus,250.0,56.0


**Domanda**: tra le distanze segnalate come anomale, quali sono probabilmente errori e quali casi reali? Guardare il mezzo e il tempo di ciascuna.

Risposta:

Decisione: i valori impossibili e i valori anomali giudicati errori vengono sostituiti con `NaN` (valore mancante), non corretti a mano: 250 km potrebbe essere 2,50 km, ma non è possibile saperlo. I casi reali restano.

- `dati.loc[condizione, "colonna"] = valore`: assegna il valore alla colonna solo nelle righe che soddisfano la condizione

In [11]:
import numpy as np

dati.loc[dati["distanza_km"] <= 0, "distanza_km"] = np.nan
dati.loc[dati["distanza_km"] > 100, "distanza_km"] = np.nan
dati.loc[dati["tempo_min"] <= 0, "tempo_min"] = np.nan
dati.loc[dati["tempo_min"] > 150, "tempo_min"] = np.nan
print(dati[["distanza_km", "tempo_min"]].describe().round(1))
controlla(dati["tempo_min"].max() <= 150 and dati["distanza_km"].min() > 0)

       distanza_km  tempo_min
count        147.0      144.0
mean           7.0       29.0
std            6.7       17.5
min            0.4        3.0
25%            2.0       15.8
50%            4.7       24.0
75%           10.2       43.2
max           35.0       75.0
Corretto.


## 5. Valori mancanti (esercizio 5)

- `isna().sum()`: numero di valori mancanti per colonna
- `dropna(subset=[...])`: elimina le righe con valori mancanti nelle colonne indicate
- `fillna(valore)`: sostituisce i valori mancanti con un valore (per esempio la mediana)

Eliminare riduce il campione; sostituire inventa un valore plausibile ma non vero. La scelta dipende dall'uso: per un grafico della distanza bastano le righe con la distanza; per un modello servono righe complete.

In [12]:
print(dati.isna().sum())

id                 0
inviato_il         0
anno_corso         0
mezzo              0
distanza_km        3
tempo_min          6
fascia_partenza    2
dtype: int64


In [13]:
completi = dati.dropna(subset=["distanza_km", "tempo_min"])
print("righe complete per distanza e tempo:", len(completi), "su", len(dati))

righe complete per distanza e tempo: 141 su 150


## 6. Salvare e documentare

- `to_csv(nome, index=False)`: salva il DataFrame in un nuovo CSV senza la colonna dell'indice
- la colonna `inviato_il` non serve più all'analisi e si elimina (minimizzazione)

In [14]:
pulito = dati.drop(columns=["inviato_il"])
pulito.to_csv("tragitti_puliti_classe.csv", index=False)
print(pulito.shape)
pulito.head()

(150, 6)


,id,anno_corso,mezzo,distanza_km,tempo_min,fascia_partenza
0,1,4,bus,10.0,46.0,7:00-7:30
1,2,5,auto,8.6,27.0,7:00-7:30
2,3,5,bici,4.7,19.0,7:30-7:45
3,4,4,monopattino,3.7,20.0,dopo le 7:45
4,5,3,piedi,0.8,11.0,dopo le 7:45


### Registro della pulizia

Completare con le operazioni svolte, il numero di righe o valori coinvolti e la motivazione.

| operazione | righe o valori coinvolti | motivazione |
|---|---|---|
| mezzo: spazi, minuscole, varianti | | |
| distanza: virgola decimale | | |
| tempo: unità nel testo | | |
| duplicati eliminati | | |
| valori impossibili resi mancanti | | |
| valori anomali resi mancanti | | |
| colonna eliminata | | |